# Sky source review

Interactive review of a sky position in OVRO-LWA time–frequency data. Enter a
**coordinate string** in the review UI — ICRS degrees (`RA, Dec`) or a source name —
then use the action buttons:

- **Center** — centers HiPS and the radio overlay on the coordinate field (typed
  name or sky click). Use after panning when you want to return to a named target.
- **Generate heatmap** — build the time–frequency heatmap for the coordinate.
- **Overlay: on/off** — toggle the radio overlay. Off hides it (HiPS only); clicking
  a heatmap cell turns the overlay back on and loads that slice. The heatmap stays
  clickable either way.

A **zeros heatmap grid** (full Zarr `time × frequency` shape) appears as soon as the
store opens, so you can **click any cell to load that slice as an overlay** centered
on the current coordinate — no need to **Generate heatmap** first. Generating a
heatmap simply replaces the zeros with computed values.

While typing a name (first character is a letter), a dropdown of matching entries from
`known_sources.yaml` appears; pick one or keep typing (tab completion logs RA/Dec).
Numeric-first input is RA/Dec.

For the active coordinate and **heatmap method**:

1. Build a time × frequency map (tracked pixel, patch statistic, patch maximum, or
   Gaussian patch fit — same options as `jupiter_flux_review.ipynb`, plus `mad`, `std`,
   `mean`, `min`).
2. **Click** a cell in the heatmap to overlay that Zarr slice in **astrowidget.SkyWidget**
   at the current view center (view-locked reprojection when HiPS is active; zoom and
   pan are preserved when changing time/frequency). Pan to align HiPS with the radio
   layer; click **Center** to put the catalog source in the middle of the field and
   reset the heatmap to zeros when moving to a new position.

Launch with: `pixi run jupyter lab`

**Run cells in order** (config → imports → helpers → class → optional Dask → launch UI).


In [1]:
# Edit before running if your paths or cuts differ.
import os
from pathlib import Path

## Checked again when the review UI starts; fix typos before launching.
ZARR_PATH = Path("/fast/claw/I-deep-Taper-Robust-0-qa11jun.zarr/")

# Optional default for the Coordinate field (edit in the review UI after launch).
COORDINATE_STRING = ""

# Fall back to NED ObjectLookup when SkyCoord.from_name fails (requires network).
USE_NED_FALLBACK = True
NED_TIMEOUT_S = 10.0

# Known source names for autocomplete. Resolved relative to cwd and notebooks/.
KNOWN_SOURCES_PATH = Path("known_sources.yaml")

PATCH_SCALE = 5.0  # patch half-width = ceil(scale * max beam FWHM in pixels)
SKY_FOV_DEG = 8.0

# Default heatmap fill: tracked pixel, patch stats, patch_max, or patch_fit
HEATMAP_METHOD = "dynamic_spectrum"
PATCH_FIT_MAX_REDUCED_CHI_SQUARED = 10.0

# Zarr read pattern (matches jupiter_flux_review.ipynb)
ZARR_LM_CHUNK = 512

# Set False to scan the image centre for the first finite time index (extra I/O)
SKIP_FIRST_VALID_SKY_SCAN = True

# Optional distributed Dask (default off — Jupiter notebook uses none)
USE_DASK_CLIENT = False
DASK_WORKERS = 6
DASK_THREADS_PER_WORKER = 6
DASK_MEMORY_LIMIT = "16GiB"

# HiPS calibration sky (Aladin Lite background). Served by the ovro-lwa-portal Jupyter
# server extension at HIPS_HTTP_PREFIX (default /calibration/hips). Restart Jupyter after
# upgrading the package if HiPS tiles 404.
HIPS_ROOT = Path("/lustre/pipeline/calibration/hips")
HIPS_BACKGROUND = HIPS_ROOT / "Blue_I_deep_Taper_Robust-0.75_Jan25.hips"
# Override with full URL if tiles are served elsewhere (set OVRO_HIPS_HTTP_BASE too).
HIPS_HTTP_PREFIX = os.environ.get("OVRO_HIPS_HTTP_BASE", "/calibration/hips")
os.environ.setdefault("OVRO_HIPS_HTTP_BASE", HIPS_HTTP_PREFIX)
os.environ.setdefault("OVRO_HIPS_ROOT", str(HIPS_ROOT))

# HiPS background display scaling (Aladin Lite setCuts).
HIPS_BACKGROUND_PERCENTILE_LOW = 2.0
HIPS_BACKGROUND_PERCENTILE_HIGH = 98.0



In [2]:
import ovro_lwa_portal as ovro
from ovro_lwa_portal.viz.source_review_app import (
    SourceReview,
    SourceReviewConfig,
    configure_source_review_notebook,
)

configure_source_review_notebook()


In [3]:
# Optional — default matches jupiter_flux_review (no distributed Client).
if USE_DASK_CLIENT:
    from dask.distributed import Client, get_client

    try:
        dask_client = get_client()
    except ValueError:
        dask_client = Client(
            n_workers=DASK_WORKERS,
            threads_per_worker=DASK_THREADS_PER_WORKER,
            processes=False,
            memory_limit=DASK_MEMORY_LIMIT,
        )
    print(dask_client)
    print(f"Dashboard: {dask_client.dashboard_link}")
else:
    print("Dask Client disabled (same default as jupiter_flux_review).")


Dask Client disabled (same default as jupiter_flux_review).


In [4]:
ovro.validate_local_zarr_store(ZARR_PATH)

review = SourceReview(
    ZARR_PATH,
    coordinate_string=COORDINATE_STRING,
    known_sources_path=KNOWN_SOURCES_PATH,
    patch_scale=PATCH_SCALE,
    sky_fov_deg=SKY_FOV_DEG,
    patch_fit_max_reduced_chi_squared=PATCH_FIT_MAX_REDUCED_CHI_SQUARED,
    heatmap_method=HEATMAP_METHOD,
    config=SourceReviewConfig(
        zarr_lm_chunk=ZARR_LM_CHUNK,
        skip_first_valid_sky_scan=SKIP_FIRST_VALID_SKY_SCAN,
        use_ned_fallback=USE_NED_FALLBACK,
        ned_timeout_s=NED_TIMEOUT_S,
        hips_root=HIPS_ROOT,
        hips_background=HIPS_BACKGROUND,
        hips_http_prefix=HIPS_HTTP_PREFIX,
        hips_background_percentile_low=HIPS_BACKGROUND_PERCENTILE_LOW,
        hips_background_percentile_high=HIPS_BACKGROUND_PERCENTILE_HIGH,
    ),
    validate_zarr=False,
)
review.panel


Column(max_width=1048, sizing_mode='stretch_width')
    [0] Row(margin=(0, 0, 8, 0), sizing_mode='stretch_width')
        [0] AutocompleteInput(case_sensitive=False, min_characters=1, name='Coordinate', placeholder='RA°, Dec° or s..., restrict=False, search_strategy='includes', sizing_mode='stretch_width')
        [1] Button(button_type='primary', name='Center', width=80)
        [2] Button(button_type='primary', name='Generate heatmap', width=150)
        [3] Toggle(button_type='success', name='Overlay: on', value=True, width=110)
        [4] Select(description='Quantity plotted i..., name='Heatmap method', options=OrderedDict({'dynamic_spec...]), value='dynamic_spectrum', width=220)
        [5] LoadingSpinner(size=24)
    [1] Markdown(str, sizing_mode='stretch_width')
    [2] IPyWidget(VBox, height=620, sizing_mode='stretch_width')
    [3] Bokeh(None, height=420, sizing_mode='stretch_width')
    [4] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] IPyWidget(HTML, height=150, sizing_mode='stretch_width')

## Notes

- **Coordinate** — enter `"RA_deg, Dec_deg"` or a source name. When the first
  character is a letter, an `includes` dropdown lists hits from `known_sources.yaml` (pick one or
  keep typing; tab completion logs RA/Dec). Click **Center** to put the catalog target in the
  field (HiPS only until heatmap load; HiPS + overlay when a slice is shown). **Pan** adjusts
  HiPS alignment — use Center, not pan, to center the source.
  (RA/Dec mode). Names resolve
  via `SkyCoord.from_name`, then NED ObjectLookup when enabled. Maps are cached
  per coordinate and heatmap method. Parse failures log a
  **WARNING** in the activity log.
- **Overlay toggle** — **Overlay: on/off** shows or hides the radio overlay without
  affecting the heatmap. With it off, clicking cells only logs the selected
  time/frequency; turn it on (or click another cell) to load the slice.
- **Click any cell** — even on the initial zeros grid, a click loads that Zarr slice
  as an overlay centered on the current coordinate. If no coordinate is set yet, the
  status prompts you to enter one (it resolves the field automatically on the click).
- **Gray heatmap cells** — NaN when the target is **outside this Zarr snapshot's field of view** at
  that time, when SKY is masked at the tracked pixel/patch, or when patch_fit is rejected by the χ² cut.
  The activity log prints a footprint hint when many cells are missing.
- **Patch size** — `PATCH_SCALE` × max beam FWHM at each time step.
- **Zarr open** — `open_dataset(..., chunks="auto").chunk({"l": 512, "m": 512})` like `jupiter_flux_review.ipynb` (`ZARR_LM_CHUNK`). Set `SKIP_FIRST_VALID_SKY_SCAN=False` only if you need a centre-pixel time scan on open.
- **Progress** — dynamic spectrum reports **Pixel track** then **Pixel I/O** (per-time reads); patch methods report in the activity log (`Patch I/O`, then `Statistics` or `Patch fit`) with time-step counts.
- First extraction on a full cube can take tens of seconds per coordinate depending on Zarr I/O and time-axis length.
- **Sky background** — OVRO-LWA calibration HiPS (`HIPS_BACKGROUND`) loads by default via
  `SkyWidget.background_survey`. Set `OVRO_HIPS_HTTP_BASE` (or edit `HIPS_HTTP_BASE`) so the
  browser can fetch tiles; Aladin Lite cannot read `/lustre/...` directly. With HiPS active,
  `overlay_view_lock` reprojects the radio overlay to the current view after pan/zoom; heatmap
  load and tap use the panned view center (catalog coordinates still drive heatmap extraction).
  Click a heatmap cell to overlay the Zarr radio slice on the background.
- **Dask (optional)** — leave `USE_DASK_CLIENT = False` unless you need the dashboard for debugging; point extractions use a local `threads` scheduler when a `Client` is active. If enabled and patch methods warn on memory, lower `DASK_WORKERS` or raise `DASK_MEMORY_LIMIT`.
